In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Understanding the Dataset

In [ ]:
df=pd.read_csv('/kaggle/input/datasets/mashlyn/online-retail-ii-uci/online_retail_II.csv')

In [ ]:
df.head(n=7)

**Business question I am trying to solve:** What are the different segments of the customers? What is their lifetime value? Which customer segments are considered high risk? and Which segments are considered valuable?

In [ ]:
df.info

In [ ]:
df.isnull().sum()

There are a lot of records that do not have customer ID.

In [ ]:
percent=(243007/df.shape[0])*100
percent

So that means around 22.76% of the data does not have Customer ID.

In [ ]:
df.describe()

Minimum quantity is -80995 which refers to the returns or cancellations.


Same is the case even with the price feature.


In [ ]:
df.duplicated().sum()

# Data Cleaning

As we found of that approx 22.76% of the data has missing customer ID, we need to remove them as we are concentrating on churn prediction, segment analysis

Similarly, we will also remove all the duplicated enteries.

We shall also drop the negative prices and negative quantities. But we will look at them seperately.


In [ ]:
df=df.dropna(subset=['Customer ID'])

In [ ]:
returnPrice_df=df[df['Price']<0]
returnQuantity_df=df[df['Quantity']<0]

In [ ]:
df=df[(df['Price']>0) & (df['Quantity']>0)]
df.shape

# Feature Engineering 

Next we have to add the revenue feature which is derieved as a product of price and quantity feature.

Similarly, we will add another column named InvoiceDate which will convert to datetime format to enable time-based analysis

In [ ]:
df['Revenue']=df['Price']*df['Quantity']
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['YearMonth'] = df['InvoiceDate'].dt.to_period('M')

df.head(n=7)

In [ ]:
print(f"\nDate range: {df['InvoiceDate'].min()} to {df['InvoiceDate'].max()}")

# Exploratory Data Analysis

## 1. Revenue analysis month wise

In [ ]:
monthly_sales = df.groupby('YearMonth')['Revenue'].sum()

monthly_sales.plot()

This shows us that the sales peaked up in the month of Nov in both the years.

December, the sales drop, this is most likely to happen in wholesale models.

Feb is the weakest months in both the years.



## 2. Top 7 Countries by revenue

In [ ]:
country_wise=df.groupby('Country')['Revenue'].sum().sort_values(ascending=False).head(n=7)
import matplotlib.pyplot as plt


plt.figure()
plt.bar(country_wise.index, country_wise.values)

plt.xticks(rotation=45)
plt.xlabel("Country")
plt.ylabel("Revenue")
plt.title("Top 10 Countries by Revenue")

plt.show()

In [ ]:
country_df = country_wise.reset_index()
country_df.columns = ['Country', 'Revenue']
total_revenue=df['Revenue'].sum()
country_df['Revenue share']=(country_df['Revenue']/total_revenue)*100
country_df

We understand that the revenue is highly concentrated in the UK market with about 83% of the total revenue.

EIRE, Netherlands, Germany are the next leading markets. France and Australia are underperforming but show potential. 

Hence, International expansion, particularly in countries like the ones mentioned above have a possibility of growth opportunity which is worth further investigation.


In [ ]:
country_revenue = df.groupby('Country')['Revenue'].sum().sort_values(ascending=False).head(10)
country_revenue_df = country_revenue.reset_index()
country_revenue_df.columns = ['Country', 'Revenue']
country_revenue_df['% of Total Revenue'] = (country_revenue_df['Revenue'] / df['Revenue'].sum() * 100).round(2)

print(country_revenue_df)

In [ ]:
import seaborn as sns
country_revenue_excl_uk = df[df['Country'] != 'United Kingdom'].groupby('Country')['Revenue'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(12, 5))
sns.barplot(x=country_revenue_excl_uk.index, y=country_revenue_excl_uk.values, color='steelblue')
plt.title('Top 10 Countries by Revenue (Excl. UK)', fontsize=14, fontweight='bold')
plt.xlabel('Country')
plt.ylabel('Revenue (£)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(country_revenue_excl_uk)

**Key Insights: Revenue by Country**

- The UK dominates revenue at £14.4M, approximately 83% of total revenue, 
  dwarfing all other markets combined.
- No international market exceeds £1M, highlighting a significant 
  concentration risk for the business.
- EIRE (£617k), Netherlands (£554k) and Germany (£425k) are the 
  leading international markets but remain relatively small.
- France and Australia show potential but are currently underperforming 
  relative to market size.
- The heavy UK dependence means any domestic demand slowdown, as 
  hinted at in late 2011, has an outsized impact on total revenue.
- International expansion, particularly in established markets like 
  Germany and France, represents an untapped growth opportunity 
  worth further investigation.

## 3. Top 10 Products by Revenue
Identifying the best performing products to understand what drives the business commercially.

**Note on data quality:** Initial results revealed two non-product entries in the top 10 
— 'Manual' (admin adjustments) and 'POSTAGE' (shipping charges). These were excluded 
from the analysis as they do not represent genuine product revenue. This is an extension 
of the cleaning process and demonstrates the importance of sense-checking results 
at every stage of analysis, not just at the start.

In [ ]:
val = df['Description'].value_counts().head(10)
print(val)

In [ ]:
non_products = ['Manual', 'POSTAGE', 'DOTCOM POSTAGE', 'Discount']

product_quantity = df[~df['Description'].isin(non_products)] \
    .groupby('Description')['Quantity'] \
    .sum() \
    .sort_values(ascending=False) \
    .head(10)

plt.figure(figsize=(14, 8))
sns.barplot(
    x=product_quantity.index,
    y=product_quantity.values,
    color='steelblue'
)

plt.title('Top 10 Products by Quantity Sold', fontsize=14, fontweight='bold')
plt.xlabel('Product')
plt.ylabel('Total Quantity')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(product_quantity)

These top items suggest the store mainly sold:

* Home decor
* Gift items
* Kitchen accessories
* Party / baking supplies
* Novelty toys
* Storage products

**Key Insights: Top 10 Products by Revenue**

- WORLD WAR 2 GLIDERS ASSTD DESIGNS is the top revenue-generating product.
- The top 10 is dominated by home décor, gift and seasonal items, confirming 
  the business operates as a gift and homewares wholesaler.
- Several top products are seasonal, which reinforces the November revenue peak identified in the monthly trend analysis. 
- A full product profitability analysis would be recommended to identify which products drive volume vs which drive margin.
- If these are top-selling by quantity, customers likely preferred low-cost repeat-purchase household and celebration items rather than expensive luxury products.

## 4. Monthly Order by volume

In [ ]:

monthly_orders = df.groupby('YearMonth')['Invoice'].nunique().reset_index()
monthly_orders.columns = ['YearMonth', 'Orders']
monthly_orders['YearMonth'] = monthly_orders['YearMonth'].astype(str)

plt.figure(figsize=(14, 5))
sns.lineplot(data=monthly_orders, x='YearMonth', y='Orders', marker='o', color='steelblue')
plt.xticks(rotation=45, ha='right')
plt.title('Monthly Order Volume — Online Retail II (2009–2011)', fontsize=14, fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Number of Orders')
plt.tight_layout()
plt.show()

**Key Insights: Monthly Order Volume**

- Order volume closely mirrors the revenue trend, confirming that revenue 
  growth is driven primarily by order volume rather than increasing order values.
- Revenue is high in the month of Nov which explains the high sales at that time.
- Despite higher February order volume, February revenue remains low — 
  indicating smaller average order values in this month versus the rest of the year.
- November remains the clear peak for both orders and revenue, 
  reinforcing the pre-Christmas stocking pattern identified earlier.

### 5. Average Order Value by Month


In [ ]:
monthly_aov = []

for month in df['YearMonth'].unique():

    month_data = df[df['YearMonth'] == month]

    invoice_total = month_data.groupby('Invoice')['Revenue'].sum()

    avg_value = invoice_total.mean()

    monthly_aov.append([str(month), avg_value])

monthly_aov = pd.DataFrame(monthly_aov, columns=['YearMonth', 'AOV'])

plt.figure(figsize=(14,5))

plt.plot(monthly_aov['YearMonth'],
         monthly_aov['AOV'],
         marker='o')

plt.xticks(rotation=45)
plt.xlabel("Month")
plt.ylabel("Average Order Value (£)")
plt.title("Average Order Value by Month")

plt.show()


- January has a high average order value in both years, likely because retailers place large restocking orders after Christmas.
- From March to July 2010, the average order value gradually decreases, showing that customers placed smaller orders during this period.
- April 2011 has the lowest average order value, meaning both the number of orders and the order sizes were low that month.
- December 2011 shows a sudden increase, but this is because the dataset only contains data until - December 9th 2011, so it should not be treated as a real trend.
- November has the highest revenue because customers placed both a large number of orders and higher-value orders before Christmas.

### 6. New vs Returning Customers by Month

In [ ]:
first_purchase = df.groupby('Customer ID')['InvoiceDate'].min()

df['FirstPurchaseDate'] = df['Customer ID'].map(first_purchase)

df['FirstPurchaseMonth'] = df['FirstPurchaseDate'].dt.to_period('M')

df['CustomerType'] = 'Returning'

df.loc[df['YearMonth'] == df['FirstPurchaseMonth'], 'CustomerType'] = 'New'

customer_type_monthly = df.groupby(
    ['YearMonth', 'CustomerType']
)['Customer ID'].nunique().reset_index()

customer_type_monthly['YearMonth'] = customer_type_monthly['YearMonth'].astype(str)

plt.figure(figsize=(14,5))

sns.lineplot(
    data=customer_type_monthly,
    x='YearMonth',
    y='Customer ID',
    hue='CustomerType',
    marker='o'
)

plt.xticks(rotation=45)

plt.xlabel("Month")
plt.ylabel("Customers")
plt.title("New vs Returning Customers by Month")

plt.show()

- Returning customers increase steadily over time and reach their highest numbers in November, showing strong customer loyalty and repeat purchases during the holiday season.
- By the end of 2011, returning customers are much higher than new customers, which suggests the business depends heavily on repeat buyers.
- New customer growth slowly decreases over time, though October shows a small increase in both years, possibly due to new retailers preparing for Christmas sales.
- April 2011 shows a drop in returning customers, matching the lower revenue and order values seen earlier, which may indicate an important business issue during that month.
- Overall, the business relies more on keeping existing customers than gaining new ones, making customer retention very important.

# RFM Analysis
RFM (Recency, Frequency, Monetary) analysis segments customers by behaviour 
to identify the most valuable customers and those at risk of churning.

- **Recency** — how recently did the customer last purchase?
- **Frequency** — how many orders have they placed?
- **Monetary** — how much have they spent in total?

Each customer is scored 1–4 on each dimension and combined into segments 
to guide commercial decision making.

In [ ]:
import datetime as dt

reference_date = df['InvoiceDate'].max() + dt.timedelta(days=1)

recency = df.groupby('Customer ID')['InvoiceDate'].max()
recency = (reference_date - recency).dt.days

frequency = df.groupby('Customer ID')['Invoice'].nunique()

monetary = df.groupby('Customer ID')['Revenue'].sum()

rfm = pd.DataFrame({
    'Recency': recency,
    'Frequency': frequency,
    'Monetary': monetary
}).reset_index()

print(rfm.shape)
print(rfm.head(10))

In [ ]:
print(f"Total unique customers: {rfm.shape[0]}")
print(f"\nRecency (days):")
print(rfm['Recency'].describe())
print(f"\nFrequency (orders):")
print(rfm['Frequency'].describe())
print(f"\nMonetary (£):")
print(rfm['Monetary'].describe())

## 1. RFM Scoring
Each customer is scored 1–4 on each RFM dimension using quartiles.
Higher scores indicate more valuable behaviour, except for Recency where 
a lower number of days is better, so the scale is inverted.

In [ ]:
rfm['R_Score'] = pd.qcut(
    rfm['Recency'],
    4,
    labels=[4, 3, 2, 1],
    duplicates='drop'
)

rfm['F_Score'] = pd.qcut(
    rfm['Frequency'].rank(method='first'),
    4,
    labels=[1, 2, 3, 4]
)

rfm['M_Score'] = pd.qcut(
    rfm['Monetary'].rank(method='first'),
    4,
    labels=[1, 2, 3, 4]
)

rfm['RFM_Score'] = (
    rfm['R_Score'].astype(str) +
    rfm['F_Score'].astype(str) +
    rfm['M_Score'].astype(str)
)

print(rfm.head(10))

## 2. Customer Segmentation
Customers are assigned to segments based on their RFM scores, 
enabling targeted commercial action for each group.

In [ ]:
def assign_segment(row):
    r = int(row['R_Score'])
    f = int(row['F_Score'])
    m = int(row['M_Score'])
    
    if r >= 3 and f >= 3 and m >= 3:
        return 'Champion'
    elif r >= 3 and f >= 2:
        return 'Loyal'
    elif r >= 3 and f <= 2:
        return 'New/Promising'
    elif r == 2 and f >= 3:
        return 'At Risk'
    elif r == 2 and f <= 2:
        return 'Needs Attention'
    else:
        return 'Lost'

rfm['Segment'] = rfm.apply(assign_segment, axis=1)

print(rfm['Segment'].value_counts())

In [ ]:
segment_counts = rfm['Segment'].value_counts().reset_index()

segment_counts.columns = ['Segment', 'Count']

plt.figure(figsize=(10,6))

plt.bar(
    segment_counts['Segment'],
    segment_counts['Count']
)

plt.xticks(rotation=45)

plt.xlabel("Segment")
plt.ylabel("Customers")
plt.title("Customer Segments RFM Analysis")

plt.show()

## 3. Revenue by Segment
Examining how revenue is distributed across segments to understand 
the commercial weight of each group.

In [ ]:
segment_revenue = rfm.groupby('Segment')['Monetary'].sum()

plt.figure(figsize=(10,6))

plt.bar(
    segment_revenue.index,
    segment_revenue.values
)

plt.xticks(rotation=45)

plt.xlabel("Segment")
plt.ylabel("Revenue")
plt.title("Revenue by Customer Segment")

plt.show()

print(segment_revenue)

- Champions make up about 31% of customers but generate the highest revenue (£13.46M), showing that the business strongly depends on these high-value customers.
- At Risk customers contribute around £1.77M in revenue. Since these customers were active before but are now purchasing less, retaining them should be a major priority.
- Lost customers account for more than £1M in revenue, suggesting that customer churn has a noticeable business impact and should be investigated further.
- Loyal customers generate around £792K in revenue and could potentially become Champion customers through targeted offers and upselling.
- Needs Attention and New/Promising customers currently contribute lower revenue, but they still represent future growth opportunities for the business.

**Main Recommendation:**
The business should focus on protecting Champion customers while also re-engaging At Risk customers to prevent future revenue loss.